# 🚀 تشغيل وكيل الذكاء الاصطناعي على Google Colab

هذا الدفتر يسمح لك بتشغيل نموذج **Qwen-0.5B-Chat** (أو أي نموذج آخر أكبر) على موارد Google Colab لتجاوز قيود الذاكرة في مساحات Hugging Face المجانية.

## ⚙️ الإعداد والتشغيل

1.  **تغيير نوع وقت التشغيل (Runtime):** تأكد من أنك تستخدم وحدة معالجة رسوميات (GPU) لتسريع تحميل النموذج. اذهب إلى `Runtime` -> `Change runtime type` واختر `T4 GPU` أو ما هو متاح.
2.  **تنفيذ الخلايا:** قم بتنفيذ الخلايا بالترتيب.
3.  **الرابط العام:** سيظهر رابط عام (Public URL) لواجهة Gradio في نهاية التنفيذ. انقر عليه للتفاعل مع الوكيل.

In [ ]:
# 1. تثبيت الاعتماديات
!pip install gradio torch transformers accelerate bitsandbytes

# 2. استيراد المكتبات
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 3. الإعدادات
MODEL_NAME = "Qwen/Qwen-0.5B-Chat" # يمكنك تغيير هذا إلى Qwen/Qwen-1.8B-Chat أو أي نموذج آخر
SYSTEM_PROMPT = "أنت مساعد ذكاء اصطناعي مفيد وودود، تجيب على الأسئلة باللغة العربية بطلاقة."

# 4. تحميل النموذج مع الكمّ (Quantization)
try:
    # استخدام 4-bit quantization لكفاءة الذاكرة
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✅ تم تحميل النموذج {MODEL_NAME} بنجاح مع 4-bit quantization.")

except Exception as e:
    print(f"❌ خطأ في تحميل النموذج {MODEL_NAME}: {e}")
    # آلية احتياطية بسيطة
    MODEL_NAME = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    SYSTEM_PROMPT = "أنت مساعد ذكاء اصطناعي بسيط. لا يمكنني معالجة اللغة العربية بشكل جيد بسبب قيود الموارد."
    print(f"⚠️ تم التحول إلى النموذج الاحتياطي {MODEL_NAME} بسبب قيود الموارد.")

In [ ]:
# 5. دالة الدردشة
def chat_with_model(message, history):
    # تنسيق سجل الدردشة لنموذج Qwen
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    # ترميز وتوليد الاستجابة
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # توليد الاستجابة مع التدفق (Streaming)
    # ملاحظة: يتم هنا توليد الاستجابة بالكامل ثم إرسالها، يمكن تعديلها لدعم التدفق الحقيقي إذا لزم الأمر
    outputs = model.generate(
        input_ids,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        repetition_penalty=1.02,
        pad_token_id=tokenizer.eos_token_id,
    )

    # فك ترميز الاستجابة وإرجاعها
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

In [ ]:
# 6. واجهة Gradio والتشغيل
iface = gr.ChatInterface(
    chat_with_model,
    title=f"وكيل الذكاء الاصطناعي (النموذج: {MODEL_NAME}) - يعمل على Colab",
    description="مساعد ذكاء اصطناعي يدعم اللغة العربية. يعمل على موارد Google Colab لتجاوز قيود الذاكرة.",
    theme="soft",
    submit_btn="إرسال",
    retry_btn="إعادة المحاولة",
    undo_btn="تراجع",
    clear_btn="مسح المحادثة",
)

# تشغيل الواجهة مع share=True للحصول على رابط عام
iface.launch(share=True)